# Evaluations: the full loop

`grasp_agents.evals` evaluates any grasp-agents `Processor` — a single agent, a workflow, a parallel fan-out — against a versioned dataset, with per-example scorers, aggregate metrics with confidence intervals, and append-only on-disk run records (a completed run's results never change) that you can compare, rescore and push to Phoenix.

| concept | what it is |
|---|---|
| **Task** | the system under test: any `Processor` (or async function), run in an isolated session per trial |
| **Dataset** / **Example** | typed examples with stable ids, a reference (ground truth), metadata and splits |
| **Scorer** → **Score** | a versioned instrument judging one trial (code, or a judge such as an LLM agent) |
| **Metric** | an aggregate over scores, with a 95% CI (repetitions are reduced per example first) |
| **Trial** | one execution of the task on one example (× repetitions) |
| **EvaluationRun** | the record of a run: provenance, trials, scores, metrics |
| **Evaluation** | a named, importable definition — what the CLI and nightly jobs run |

The demo component is a **short-answer grader** for a tutoring product: a two-step `SequentialWorkflow` that checks which key points of a reference answer a student's answer covers, then writes a verdict and feedback. `v1` is a naive first attempt; `v2` handles paraphrases and negations. The writer flips borderline verdicts at random to behave like a sampled model. Everything runs offline; section 13 swaps in a real LLM.

The loop: **validate data → baseline → read failures → fix → compare → re-judge → A/B → held-out check → validate the judge → push**.

In [1]:
import os
import sys
import tempfile
from pathlib import Path

from IPython.display import Markdown, display

import grasp_agents.examples.evals.grader_evals as demo  # pipeline, data, scorers, evaluations

DEMO_DIR = Path(demo.__file__).parent

from grasp_agents.evals import (
    Dataset,
    LocalRunStore,
    compare,
    example_json_schema,
    pairwise,
    render_comparison_markdown,
    render_run_markdown,
)

# Runs normally go to ./.evals (or $GRASP_EVALS_DIR); keep the demo's apart.
EVALS_DIR = Path(tempfile.mkdtemp(prefix="grasp-evals-demo-"))
store = LocalRunStore(EVALS_DIR)

## 1. The component under test

A plain grasp-agents pipeline — nothing about it knows it is being evaluated.

In [2]:
grader = demo.build_grader("v1", noise=0.0)
submission = demo.Submission(
    question="What do plants need for photosynthesis?",
    reference_answer="Sunlight, water and carbon dioxide.",
    student_answer="Plants do not need sunlight, just water and carbon dioxide.",
)
packet = await grader.run(in_args=submission)
packet.payloads[0]

Grade(verdict='correct', feedback='Good job!')

## 2. The dataset

Typed examples (`Submission` in, the teacher's `TeacherGrade` as reference) with stable ids, `dev` / `test` splits and strata in metadata. The **fingerprint** hashes the content, so every run records exactly what it evaluated. `test` is **sealed**: reports, `show`, `compare` and Phoenix pushes give it only in aggregate, and a run that touches it must include all of it — so whoever iterates on the grader through these tools, a person or a coding agent, cannot fit to its examples. (It is not access control: the data file contains it.)

In [3]:
dataset = Dataset.load(demo.DATA, input_type=demo.Submission, reference_type=demo.TeacherGrade)
print(f"{len(dataset)} examples, fingerprint {dataset.fingerprint}")
print({split: len(dataset.split(split)) for split in dataset.splits})
example = dataset["bio-3"]
print(example.input.student_answer, "->", example.reference)

24 examples, fingerprint c7c7adb4b34d
{'dev': 16, 'test': 8}
Plants do not need sunlight, just water and carbon dioxide. -> verdict='incorrect' key_issue='sunlight'


Integrity checks run before every run (here: a correct answer cannot have a key issue). Agents authoring new examples get a JSON Schema for the record format:

In [4]:
print("problems:", demo.grader_v1.check_dataset(dataset))
schema = example_json_schema(demo.Submission, demo.TeacherGrade)
print(sorted(schema["properties"]), "required:", schema["required"])

problems: []
['id', 'input', 'metadata', 'reference', 'splits'] required: ['input']


## 3. Scorers

Scorers are versioned, task-specific instruments. Their outcomes are kept apart — a failure never becomes a number:

- a returned value is a **score** (`bool` = pass/fail, number, or `str` label);
- `Score.unscored(reason)` is **unscored** (e.g. an unparseable judge verdict) — excluded from metrics and counted;
- returning `None` means **not applicable** (here: the example has no key issue to name);
- an exception is an **scorer failure**; a task exception or timeout is a **task error**.

In [5]:
import inspect

print(inspect.getsource(demo.names_key_issue._fn))

@scorer(version="1")
def names_key_issue(ctx: Ctx) -> Score | None:
    """Does the feedback name what is wrong or missing? N/A when nothing is."""
    if ctx.reference is None or ctx.reference.key_issue is None:
        return None
    issue = ctx.reference.key_issue
    hit = issue.lower() in ctx.output.feedback.lower()
    return Score(
        name="names_key_issue",
        value=hit,
        explanation=f"expected '{issue}' in: {ctx.output.feedback}",
    )



## 4. Baseline

Each example runs **3 times** (the writer is noisy, like a sampled model). Metrics reduce repetitions per example first, so `n` counts examples, not trials; `pass^3` is the chance that all three runs agree with the teacher — the reliability a learner experiences.

In [6]:
run_v1 = await demo.grader_v1.run(split="dev", store=store)
display(Markdown(render_run_markdown(run_v1)))

# short-answer-grader — completed

- **Run:** `20261001T151014-short-answer-grader-65ce` (evaluation) · 2026-10-01 15:10 UTC
- **Question:** Does the grader agree with teachers and explain mistakes?
- **Task:** grader@v1 (`grasp_agents.workflow.sequential_workflow.SequentialWorkflow`)
- **Dataset:** short_answers `c7c7adb4b34d` · 16/24 examples — split=dev
- **Scorers:** agrees_with_teacher@1 (CODE), names_key_issue@1 (CODE), feedback_concise@1 (CODE), feedback_quality@1 (CODE)
- **Code:** `86b84dccec36` (grasp-evals-judges) +uncommitted changes
- **Trials:** 48/48 (3 per example) · task errors 0 · scorer failures 0 · unscored 0

## Metrics

| metric | value | 95% CI | n | missing | n/a |
|---|---|---|---|---|---|
| pass_rate(agrees_with_teacher) | 0.458 | [0.249, 0.684] | 16 | 0 | 0 |
|   ↳ difficulty=easy | 0.667 | [0.392, 0.861] | 11 | 0 | 0 |
|   ↳ difficulty=hard | 0 | [0, 0.434] | 5 | 0 | 0 |
| pass^3(agrees_with_teacher) | 0.312 | [0.128, 0.584] | 16 | 0 | 0 |
|   ↳ difficulty=easy | 0.455 | [0.182, 0.757] | 11 | 0 | 0 |
|   ↳ difficulty=hard | 0 | [0, 0.607] | 5 | 0 | 0 |
| pass_rate(names_key_issue) | 0 | [0, 0.324] | 8 | 0 | 8 |
|   ↳ difficulty=easy | 0 | [0, 0.561] | 3 | 0 | 8 |
|   ↳ difficulty=hard | 0 | [0, 0.434] | 5 | 0 | 0 |
| pass_rate(feedback_concise) | 1 | [0.806, 1] | 16 | 0 | 0 |
|   ↳ difficulty=easy | 1 | [0.741, 1] | 11 | 0 | 0 |
|   ↳ difficulty=hard | 1 | [0.566, 1] | 5 | 0 | 0 |
| pass_rate(feedback_quality) | 0.5 | [0.288, 0.712] | 16 | 0 | 0 |
|   ↳ difficulty=easy | 0.485 | [0.238, 0.739] | 11 | 0 | 0 |
|   ↳ difficulty=hard | 0.533 | [0.122, 0.904] | 5 | 0 | 0 |
| p95(duration_s) | 0.000585 | - | 48 | 0 | 0 |
|   ↳ difficulty=easy | 0.000683 | - | 33 | 0 | 0 |
|   ↳ difficulty=hard | 0.000458 | - | 15 | 0 | 0 |

## Lowest `agrees_with_teacher`

| example | mean | reps | explanation (worst repetition) |
|---|---|---|---|
| `bio-2` | 0 | 3 | teacher: correct, grader: incorrect |
| `bio-3` | 0 | 3 | teacher: incorrect, grader: correct |
| `phy-3` | 0 | 3 | teacher: incorrect, grader: partial |
| `phy-4` | 0 | 3 | teacher: incorrect, grader: correct |
| `math-2` | 0 | 3 | teacher: correct, grader: incorrect |

## Lowest `feedback_quality`

| example | mean | reps | explanation (worst repetition) |
|---|---|---|---|
| `bio-1` | 0 | 3 | Good job! |
| `bio-3` | 0 | 3 | Good job! |
| `phy-1` | 0 | 3 | Good job! |
| `phy-4` | 0 | 3 | Good job! |
| `chem-1` | 0 | 3 | Good job! |

## Lowest `names_key_issue`

| example | mean | reps | explanation (worst repetition) |
|---|---|---|---|
| `bio-3` | 0 | 3 | expected 'sunlight' in: Good job! |
| `bio-4` | 0 | 3 | expected 'carbon' in: Some points are missing. |
| `phy-3` | 0 | 3 | expected 'gravity' in: Some points are missing. |
| `phy-4` | 0 | 3 | expected 'attract' in: Good job! |
| `math-3` | 0 | 3 | expected 'half' in: Some points are missing. |


## 5. Read the failures

The headline number is only a pointer; the examples behind it are the evidence.

In [7]:
for trial in run_v1.trials:
    agree = trial.score("agrees_with_teacher")
    if trial.repetition == 0 and agree is not None and agree.value is False:
        student = run_v1.example(trial.example_id).input.student_answer
        print(f"{trial.example_id:7} {agree.explanation:38} | {student}")

bio-2   teacher: correct, grader: incorrect    | They need sun, H2O and CO2.
bio-3   teacher: incorrect, grader: correct    | Plants do not need sunlight, just water and carbon dioxide.
phy-2   teacher: correct, grader: partial      | Because gravity pulls things down to earth.
phy-3   teacher: incorrect, grader: partial    | Objects fall because the Earth spins.
phy-4   teacher: incorrect, grader: correct    | Gravity does not attract objects; they just fall toward Earth.
math-2  teacher: correct, grader: incorrect    | A half, since each half piece is bigger than a third.
math-3  teacher: incorrect, grader: partial    | One third is larger because three pieces are more than two.
chem-3  teacher: incorrect, grader: partial    | It is made of hydrogen and carbon.


v1 misses paraphrases ("sun, H2O and CO2") and negations ("do not need sunlight"), and its feedback never names the problem. `v2` adds synonyms, stemming and negation handling.

## 6. Fix, rerun, compare

The comparison pairs examples by id (and unchanged content) and reports the **paired** difference with a CI and p-value, the **minimum detectable effect** for this dataset size, and the examples that moved.

In [8]:
run_v2 = await demo.grader_v2.run(split="dev", store=store)
display(Markdown(render_comparison_markdown(compare(run_v1, run_v2))))

# short-answer-grader vs short-answer-grader

- **Base:** `20261001T151014-short-answer-grader-65ce` · **Candidate:** `20261001T151014-short-answer-grader-f1ae`
- **Paired examples:** 16 (changed content 0, base-only 0, candidate-only 0)

Δ = candidate - base on paired examples; ✱ = significant at alpha = 0.05 after Holm correction (outcomes and resources separately; p is unadjusted); MDE = smallest true Δ detectable at 80% power.

| target | base | candidate | Δ | CI | p | MDE | n | +/- |
|---|---|---|---|---|---|---|---|---|
| ↑ agrees_with_teacher | 0.458 | 0.75 | 0.292 ✱ | [0.068, 0.515] | 0.014 | 0.314 | 16 | +6/-0 |
| ↑ feedback_concise | 1 | 1 | 0 | [-0.194, 0.194] | 1 | - | 16 | +0/-0 |
| ↑ feedback_quality | 0.5 | 1 | 0.5 ✱ | [0.266, 0.734] | 0.000378 | 0.329 | 16 | +11/-0 |
| ↑ names_key_issue | 0 | 0.875 | 0.875 ✱ | [0.731, 1] | 1.9e-06 | 0.199 | 8 | +8/-0 |
| ↓ error | 0 | 0 | 0 | [-0.194, 0.194] | 1 | - | 16 | +0/-0 |
| ↓ duration_s | 0.000476 | 0.000482 | 6.53e-06 | [-2.16e-05, 3.47e-05] | 0.628 | 3.96e-05 | 16 | +4/-12 |


## 7. Change the instrument, not the task: rescoring

A stricter `feedback_quality` judge (version 2) re-scores v2's **stored outputs** — the task does not run again. The result is a child run; the parent is never modified. Unchanged scorers keep their scores — only the changed one runs again — and the comparison warns that the scorer version changed.

In [9]:
strict = await demo.grader_v2_strict.rescore(run_v2, store=store)
print("child of", strict.parent_run_id)
display(Markdown(render_comparison_markdown(compare(run_v2, strict))))

child of 20261001T151014-short-answer-grader-f1ae


# short-answer-grader vs short-answer-grader

- **Base:** `20261001T151014-short-answer-grader-f1ae` · **Candidate:** `20261001T151014-short-answer-grader-9724`
- **Paired examples:** 16 (changed content 0, base-only 0, candidate-only 0)
- ⚠ Scorer 'feedback_quality' changed version (1 → 2): score differences may come from the instrument.

Δ = candidate - base on paired examples; ✱ = significant at alpha = 0.05 after Holm correction (outcomes and resources separately; p is unadjusted); MDE = smallest true Δ detectable at 80% power.

| target | base | candidate | Δ | CI | p | MDE | n | +/- |
|---|---|---|---|---|---|---|---|---|
| ↑ agrees_with_teacher | 0.75 | 0.75 | 0 | [0, 0] | 1 | - | 16 | +0/-0 |
| ↑ feedback_concise | 1 | 1 | 0 | [-0.194, 0.194] | 1 | - | 16 | +0/-0 |
| ↑ feedback_quality | 1 | 0.938 | -0.062 | [-0.134, 0.009] | 0.083 | 0.101 | 16 | +0/-3 |
| ↑ names_key_issue | 0.875 | 0.875 | 0 | [0, 0] | 1 | - | 8 | +0/-0 |
| ↓ error | 0 | 0 | 0 | [-0.194, 0.194] | 1 | - | 16 | +0/-0 |
| ↓ duration_s | 0.000482 | 0.000482 | 0 | [0, 0] | 1 | - | 16 | +0/-0 |

## Regressions in `feedback_quality`

| example | base | candidate | candidate explanation (worst repetition) |
|---|---|---|---|
| `math-3` | 1 | 0.667 | mentions a key point: False — Correct and complete. |
| `chem-3` | 1 | 0.667 | mentions a key point: False — Correct and complete. |
| `chem-4` | 1 | 0.667 | mentions a key point: False — Correct and complete. |


## 8. Pairwise A/B

A pairwise judge sees two outputs as "first" and "second" — never which version produced them — and is asked in **both orders**, so position bias shows up as inconsistency. The result is itself a run, with a win rate, its CI and a sign test over examples; an arm whose task failed on an example loses that pair.

In [10]:
ab = await pairwise(
    run_v1,
    run_v2,
    demo.specific_feedback_judge,
    input_type=demo.Submission,
    reference_type=demo.TeacherGrade,
    output_type=demo.Grade,
    store=store,
)
win = ab.metric("win_rate(specificity)")
print(f"v2 wins {win.value:.0%} [{win.ci_low:.0%}, {win.ci_high:.0%}], details: {win.details}")
print("position-consistent:", ab.metric("pass_rate(specificity.position_consistent)").value)

v2 wins 74% [59%, 85%], details: {'candidate': 23, 'base': 0, 'tie': 25, 'inconsistent': 0, 'examples_won': 10, 'examples_lost': 0, 'sign_test_p': 0.001953125}
position-consistent: 1.0


## 9. The held-out test split

Only aggregates are shown for sealed splits — no per-example tables — and the run includes the whole split.

In [11]:
run_test = await demo.grader_v2.run(split="test", store=store)
display(Markdown(render_run_markdown(run_test)))

# short-answer-grader — completed

- **Run:** `20261001T151014-short-answer-grader-76a7` (evaluation) · 2026-10-01 15:10 UTC
- **Question:** Does the grader agree with teachers and explain mistakes?
- **Task:** grader@v2 (`grasp_agents.workflow.sequential_workflow.SequentialWorkflow`)
- **Dataset:** short_answers `c7c7adb4b34d` · 8/24 examples — split=test
- **Scorers:** agrees_with_teacher@1 (CODE), names_key_issue@1 (CODE), feedback_concise@1 (CODE), feedback_quality@1 (CODE)
- **Code:** `86b84dccec36` (grasp-evals-judges) +uncommitted changes
- **Trials:** 24/24 (3 per example) · task errors 0 · scorer failures 0 · unscored 0
- **Sealed:** 24 trials in held-out splits are reported in aggregate only

## Metrics

| metric | value | 95% CI | n | missing | n/a |
|---|---|---|---|---|---|
| pass_rate(agrees_with_teacher) | 0.875 | [0.529, 0.978] | 8 | 0 | 0 |
| pass^3(agrees_with_teacher) | 0.875 | [0.432, 0.985] | 8 | 0 | 0 |
| pass_rate(names_key_issue) | 1 | [0.439, 1] | 3 | 0 | 5 |
| pass_rate(feedback_concise) | 1 | [0.676, 1] | 8 | 0 | 0 |
| pass_rate(feedback_quality) | 1 | [0.676, 1] | 8 | 0 | 0 |
| p95(duration_s) | 0.000803 | - | 24 | 0 | 0 |


## 10. Judges: validate before you trust

`feedback_quality` so far was code standing in for a judge. A real judge — an `LLMAgent` that reads the output and returns a verdict — is an instrument whose own errors need measuring. `ProcessorScorer` turns any processor into a scorer: each call runs in isolation, its model usage counts toward the trial, and its prompt, model, settings and code are its identity. The demo's `FeedbackJudge` is an offline stand-in: v1 passes any feedback of three words or more, v2 wants feedback on an imperfect answer to name a key point, and both flip a tenth of their verdicts, like a sampled model.

**Validation is an evaluation**: the judge becomes the task, applied to stored outputs a teacher labeled (`data/feedback_labels.jsonl`, sampled from v1 and v2 runs), and the metrics are agreement — accuracy, Cohen's κ, TPR, TNR, the confusion matrix and, with repetitions, self-consistency. Iterate on `dev`; the labels' `test` split is sealed.

In [12]:
AGREEMENT = ["accuracy", "kappa", "tpr", "tnr", "consistency"]


def agreement(run):
    cells = []
    for name in AGREEMENT:
        m = run.metric(f"{name}(feedback_quality)")
        cells.append(f"{name} {m.value:.2f} [{m.ci_low:.2f}, {m.ci_high:.2f}]")
    return f"{run.task.name}@{run.task.version}: " + ", ".join(cells)


val_v1 = await demo.judge_v1_validation.run(split="dev", store=store)
val_v2 = await demo.judge_v2_validation.run(split="dev", store=store)
print(agreement(val_v1))
print(agreement(val_v2))
display(Markdown(render_comparison_markdown(compare(val_v1, val_v2))))

feedback_quality@v1: accuracy 0.40 [0.27, 0.55], kappa -0.27 [-0.54, 0.04], tpr 0.56 [0.37, 0.73], tnr 0.19 [0.05, 0.50], consistency 0.62 [0.34, 0.84]
feedback_quality@v2: accuracy 0.73 [0.50, 0.88], kappa 0.45 [-0.01, 0.75], tpr 0.75 [0.52, 0.89], tnr 0.70 [0.31, 0.93], consistency 0.52 [0.23, 0.80]


# feedback_quality-validation vs feedback_quality-validation

- **Base:** `20261001T151015-feedback-quality-validation-45d5` · **Candidate:** `20261001T151015-feedback-quality-validation-9148`
- **Paired examples:** 21 (changed content 0, base-only 0, candidate-only 0)
- **Clustered by:** `example_id`

Δ = candidate - base on paired examples; ✱ = significant at alpha = 0.05 after Holm correction (outcomes and resources separately; p is unadjusted); MDE = smallest true Δ detectable at 80% power.

| target | base | candidate | Δ | CI | p | MDE | n | +/- |
|---|---|---|---|---|---|---|---|---|
| ↑ feedback_quality.agrees | 0.397 | 0.73 | 0.333 ✱ | [0.121, 0.546] | 0.008 | 0.293 | 21 | +13/-3 |
| ↓ error | 0 | 0 | 0 | [0, 0] | 1 | - | 21 | +0/-0 |
| ↓ duration_s | 0.000535 | 0.000548 | 1.33e-05 | [-1.35e-05, 4.02e-05] | 0.278 | 3.7e-05 | 21 | +9/-12 |

## Regressions in `feedback_quality.agrees`

| example | base | candidate | candidate explanation (worst repetition) |
|---|---|---|---|
| `4a79a29633b7` | 1 | 0.333 | judge: false, label: true — the answer was correct (on a second reading: the opposite) |
| `78262488a405` | 1 | 0.333 | judge: false, label: true — the answer was correct (on a second reading: the opposite) |
| `44eb789ed2e1` | 0.333 | 0 | judge: true, label: false — the answer was correct |


The disagreements are where the next judge version comes from: read them, change the prompt (or the code), and the new version is validated from scratch — by hand, or by the agent running the loop. No automatic prompt optimizer: fitted to a few dozen labels, it would learn the labels, not the task.

In [13]:
for trial in val_v2.trials:
    agrees = trial.score("feedback_quality.agrees")
    if trial.repetition == 0 and agrees is not None and agrees.value is False:
        item = val_v2.example(trial.example_id).input
        print(f"{item.output.feedback!r:45} {agrees.explanation}")

'Some points are missing.'                    judge: true, label: false — the answer was correct
'This is not right, please review the material.' judge: true, label: false — the answer was correct
'Some points are missing.'                    judge: true, label: false — the answer was correct
'Correct and complete.'                       judge: false, label: true — the answer was correct (on a second reading: the opposite)
'Correct and complete.'                       judge: false, label: true — the answer was correct (on a second reading: the opposite)
'Correct and complete.'                       judge: true, label: false — names no key point (on a second reading: the opposite)
'You missed: half, halves.'                   judge: false, label: true — names ['half', 'halves'] (on a second reading: the opposite)


**Probes** need no labels: a degradation must lower the verdict (here: replacing the feedback on an imperfect answer with "Please review the material."), an irrelevant change must not move it (padding, upper-casing).

In [14]:
for evaluation in (demo.judge_v1_probes, demo.judge_v2_probes):
    probes = await evaluation.run(store=store)
    rates = ", ".join(
        f"{m.name.split('.')[-1].rstrip(')')} {m.value:.2f}"
        for m in probes.metrics
        if m.name.startswith(("sensitivity", "invariance"))
    )
    print(f"{probes.task.name}@{probes.task.version}: {rates}")

feedback_quality@v1: generic 0.00, padded 0.66, uppercase 0.76
feedback_quality@v2: generic 1.00, padded 0.83, uppercase 0.80


**Gate**: `grader_v2_judged` grades with the v2 judge and declares `validation_gates={"feedback_quality": ValidationGate(min_kappa=0.2, labels="feedback_labels")}`. It refuses to score until a validation of the judge *exactly as it is now* — prompt, model, settings, hooks, tools and code — has passed the gate on the sealed test split of those labels: the lower end of κ's interval must clear 0.2, and every labeled item needs a verdict. The interval is wide because the 20 test labels come from a few original examples (it is clustered on them). Once the gate passes, the run also reports the pass rate **corrected** for the judge's measured TPR and TNR.

In [15]:
from grasp_agents.evals import UnvalidatedJudgeError

try:
    await demo.grader_v2_judged.run(split="dev", store=store)
except UnvalidatedJudgeError as exc:
    print(exc)

val_test = await demo.judge_v2_validation.run(split="test", store=store)
judged = await demo.grader_v2_judged.run(split="dev", store=store)
for name in ["pass_rate(feedback_quality)", "corrected_pass_rate(feedback_quality)"]:
    m = judged.metric(name)
    print(f"{name}: {m.value:.2f} [{m.ci_low:.2f}, {m.ci_high:.2f}]")
print("validated by", judged.metadata["judge_validations"]["feedback_quality"]["run_id"])

short-answer-grader: judges are not validated:
  feedback_quality: no finished validation run of the current judge on a sealed split of 'feedback_labels' (any change to its prompt, model, settings, version or code needs a new one)
Run their validation evaluations first, or allow unvalidated judges.


pass_rate(feedback_quality): 0.94 [0.82, 0.98]
corrected_pass_rate(feedback_quality): 0.96 [0.70, 1.00]
validated by 20261001T151015-feedback-quality-validation-dac9


**More labels**: `sample_for_labeling` picks outputs to label — where the judge and a cheap check disagree first, then both verdicts evenly — never from sealed trials, never twice, and without the judge's verdicts (blind). Someone fills in each `reference`; `import_labels` (or `merge_labels` for records in memory) adds them to the labels dataset. From the shell: `grasp-evals labels sample | import | pull | show`; `pull` reads human annotations made in Phoenix on traced trials.

In [16]:
import json
import shutil

from grasp_agents.evals import sample_for_labeling
from grasp_agents.evals.labeling import merge_labels

labeled = Dataset.load(demo.LABELS)
requests = sample_for_labeling(
    judged, 5, score="feedback_quality", against="names_key_issue", exclude=labeled.ids
)
print(json.dumps(requests[0], indent=1))

labels = EVALS_DIR / "labels.jsonl"
shutil.copy(demo.LABELS, labels)
# A teacher would decide each one; here every feedback is marked specific.
print(merge_labels([{**r, "reference": True} for r in requests], labels, labeler="demo"))

{
 "id": "5771ff26d50d",
 "input": {
  "input": {
   "question": "What is water made of?",
   "reference_answer": "Hydrogen and oxygen atoms.",
   "student_answer": "Oxygen."
  },
  "output": {
   "verdict": "correct",
   "feedback": "Correct and complete."
  },
  "reference": {
   "verdict": "partial",
   "key_issue": "hydrogen"
  },
  "metadata": {
   "topic": "chemistry",
   "difficulty": "easy"
  }
 },
 "reference": null,
 "metadata": {
  "example_id": "chem-4",
  "repetition": 1,
  "run_id": "20261001T151015-short-answer-grader-5d90",
  "score": "feedback_quality"
 },
 "splits": [
  "dev"
 ]
}
path='/tmp/claude-501/grasp-evals-demo-9o6hvu9a/labels.jsonl' added=1 updated=0 unchanged=0 unlabeled=0 conflicts=[] total=42


## 11. The same loop from the shell

Everything above is also a CLI (`grasp-evals`, or `python -m grasp_agents.evals`) with `--json` output and gating exit codes — the interface a coding agent uses. `walkthrough.sh` next to the demo runs the whole loop.

In [17]:
import json
import subprocess


def evals(*args: str) -> subprocess.CompletedProcess[str]:
    env = {**os.environ, "GRASP_EVALS_DIR": str(EVALS_DIR)}
    return subprocess.run(
        [sys.executable, "-m", "grasp_agents.evals", *args],
        capture_output=True, text=True, env=env,
    )


print(evals("runs").stdout)

# What an agent runs in its loop: a small, fast slice, machine-readable, gated.
SPEC = f"{DEMO_DIR / 'grader_evals.py'}:grader_v2"
result = evals(
    "run", SPEC, "--split", "dev", "--limit", "4", "-r", "1", "-q", "--json",
    "--fail-under", "pass_rate(agrees_with_teacher)=0.5",
)
summary = json.loads(result.stdout)
print("exit code:", result.returncode, "| gate failures:", summary["gate_failures"])
print(json.dumps(summary["metrics"]["pass_rate(agrees_with_teacher)"], indent=2))

20261001T151015-short-answer-grader-5d90  evaluation short-answer-grader@v2  completed  48 trials  pass_rate(agrees_with_teacher)=0.75
20261001T151015-feedback-quality-validation-dac9  evaluation feedback_quality-validation@v2  completed  60 trials  accuracy(feedback_quality)=0.883
20261001T151015-feedback-quality-probes-a8f1  evaluation feedback_quality-probes@v2  completed  41 trials  sensitivity(feedback_quality.generic)=1
20261001T151015-feedback-quality-probes-e0e5  evaluation feedback_quality-probes@v1  completed  41 trials  sensitivity(feedback_quality.generic)=0
20261001T151015-feedback-quality-validation-9148  evaluation feedback_quality-validation@v2  completed  63 trials  accuracy(feedback_quality)=0.73
20261001T151015-feedback-quality-validation-45d5  evaluation feedback_quality-validation@v1  completed  63 trials  accuracy(feedback_quality)=0.397
20261001T151014-short-answer-grader-76a7  evaluation short-answer-grader@v2  completed  24 trials  pass_rate(agrees_with_teacher

exit code: 0 | gate failures: []
{
  "value": 1.0,
  "ci": [
    0.5101091635454028,
    1.0
  ],
  "confidence": 0.95,
  "n": 4,
  "missing": 0,
  "na": 0,
  "groups": {
    "difficulty=easy": {
      "value": 1.0,
      "ci": [
        0.4385029682449546,
        1.0
      ],
      "confidence": 0.95,
      "n": 3,
      "missing": 0,
      "na": 0
    },
    "difficulty=hard": {
      "value": 1.0,
      "ci": [
        0.2065493143772375,
        1.0
      ],
      "confidence": 0.95,
      "n": 1,
      "missing": 0,
      "na": 0
    }
  }
}


## 12. Phoenix (optional)

Runs stay canonical on disk; Phoenix is the shared store for datasets and the UI. A finished run is pushed as one experiment (one Phoenix run per trial, one evaluation per score); pushing again adds nothing. Set `PHOENIX_BASE_URL` (and `PHOENIX_API_KEY`) to try it.

In [18]:
if os.environ.get("PHOENIX_BASE_URL"):
    from grasp_agents.evals.phoenix import PhoenixClient, experiment_url, push_run

    async with PhoenixClient() as client:
        link = await push_run(client, run_v2, store=store)
    print("experiment:", experiment_url(link))
else:
    print("PHOENIX_BASE_URL not set; skipping the push")

PHOENIX_BASE_URL not set; skipping the push


## 13. Swap in a real model

The task is any `Processor`, so an `LLMAgent` drops in unchanged; per-agent token usage, cost and the models observed are recorded on every trial. Use a cheap model.

In [19]:
if os.environ.get("RUN_REAL_LLM"):
    from grasp_agents.llm_providers.gemini.gemini_llm import GeminiLLM

    llm = GeminiLLM(model_name="gemini-3.5-flash-lite", apply_output_schema_via_provider=True)
    llm_evaluation = demo.llm_grader_evaluation(llm)
    run_llm = await llm_evaluation.run(split="dev", repetitions=1, store=store)
    display(Markdown(render_run_markdown(run_llm)))
else:
    print("set RUN_REAL_LLM=1 (and a Gemini API key) to run the LLM grader")

set RUN_REAL_LLM=1 (and a Gemini API key) to run the LLM grader
